In [ ]:
%matplotlib inline


# Learning and inference of latents and of a hidden Markov model for replay characterisation in the striatum

The code below learns and infers latents and a hidden Markov models for the
characterisation of replay in the striatum


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import gcnu_common.utils.neural_data_analysis
import ssm.inference
import ssm.learning
import ssm.neural_latents.utils
import ssm.neural_latents.plotting

## Define variables



In [ ]:
# data
bin_size = 0.2  # 200~ms
sqrt_transform = True
units_info_filename = "/ceph/sjones/projects/sequence_squad/organised_data/animals/EJT178_implant1/recording6_29-03-2022/ephys/non_curated_spikes/good_units_df.csv"
spikes_times_col_name = "Spike_times"

# for EJT178_implant1/recording6_29-03-2022 clusters 0-207 belong to the striatum
clusters_indices = np.arange(208)

# EM
n_latents = 5
n_clusters = clusters_indices.size
EM_start_time_sec = 4045.0
EM_duration_sec = 300.0
EM_max_iter = 5000
EM_tol = 1e-1
EM_constraint_diag_R = True

vars_to_estimate = {
    "u": False,
    "B": True,
    "Q": True,
    "a": True,
    "Z": True,
    "R": True,
    "m0": True,
    "V0": True,
}

# EM initial conditions
constant_u = 0.0
mean_B = 0.1
sigma_B = 0.3
mean_Q = 5.0
sigma_Q = 0.5
mean_a = 1.0
sigma_a = 0.1
mean_Z = 0.01
sigma_Z = 0.005
mean_R = 1.0
sigma_R = 0.2
constant_m0 = 0.0
mean_V0 = 0.02
sigma_V0 = 0.005

u0 = constant_u * np.ones(shape=n_latents)
B0 = np.random.normal(loc=mean_B, scale=sigma_B, size=(n_latents, n_latents))
a0 = np.random.normal(loc=mean_a, scale=sigma_a, size=n_clusters)
Z0 = np.random.normal(loc=mean_Z, scale=sigma_Z, size=(n_clusters, n_latents))
Q0 = np.diag(np.abs(np.random.normal(loc=mean_Q, scale=sigma_Q,
                                     size=n_latents)))
R0 = np.diag(np.abs(np.random.normal(loc=mean_R, scale=sigma_R,
                                     size=n_clusters)))
m0_0 = constant_m0 * np.ones(shape=n_latents)
V0_0 = np.diag(np.abs(np.random.normal(loc=mean_V0, scale=sigma_V0,
                                       size=n_latents)))

## Bin spikes times



In [ ]:
# read data
units_info_df = pd.read_csv(units_info_filename)

# n_clusters
n_clusters = units_info_df.shape[0]

# get spikes timestamps
spikes_timestamps = [None for n in range(n_clusters)]
for n in range(n_clusters):
    unit_spikes_times_str = units_info_df.iloc[n][spikes_times_col_name][1:-1].split(",")
    unit_spikes_times = np.array([float(unit_spike_times_str) for unit_spike_times_str in unit_spikes_times_str])
    spikes_timestamps[n] = unit_spikes_times

# bin spikes
t_min = min(min(st) for st in spikes_timestamps if len(st) > 0)
t_max = max(max(st) for st in spikes_timestamps if len(st) > 0)
bins_edges = np.arange(t_min, t_max, bin_size)
bins_centers = (bins_edges[1:] + bins_edges[:-1])/2

binned_spikes = np.empty(shape=(n_clusters, len(bins_centers)))
for n in range(n_clusters):
    binned_spikes[n, :] = gcnu_common.utils.neural_data_analysis.binSpikesTimes(
        spikes_times=spikes_timestamps[n], bins_edges=bins_edges,
        time_unit="sec")
if sqrt_transform:
    binned_spikes = np.sqrt(binned_spikes)

## Perform EM



In [ ]:
EM_data = binned_spikes.T
EM_end_time_sec = EM_start_time_sec + EM_duration_sec
valid_bins_mask = np.logical_and(EM_start_time_sec <= bins_centers,
                                 bins_centers < EM_end_time_sec)
EM_data = EM_data[valid_bins_mask, :]

# make sure that the first EM_data point is not NaN
first_not_nan_index = np.where(~np.isnan(EM_data).any(axis=1))[0][0]
EM_data = EM_data[first_not_nan_index:,clusters_indices]

optim_res = ssm.learning.em_SS_LDS(
    y=EM_data.T, u0=u0, B0=B0, Q0=Q0, a0=a0, Z0=Z0, R0=R0,
    m0_0=m0_0, V0_0=V0_0, max_iter=EM_max_iter, tol=EM_tol,
    vars_to_estimate=vars_to_estimate,
    constraint_diag_R=EM_constraint_diag_R,
)